In [36]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [37]:
from google.colab import files

uploaded = files.upload()

KeyboardInterrupt: 

In [38]:
df = pd.read_csv("/content/placement_predict_50k Dataset.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (50000, 32)


,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly,Salary Package
0,1,Male,Ahmedabad,Tier2,ECE,Networking,No,No,6.02,6.54,...,0,66.7,2.2,49.4,47.8,0,Low,0,0,0.00
1,2,Female,Mumbai,Tier2,ECE,DataScience,Yes,Yes,5.84,5.12,...,0,48.2,2.4,26.7,25.8,0,Low,0,0,0.00
2,3,Male,Kolkata,Tier2,IT,DataScience,Yes,No,4.91,5.29,...,0,73.8,2.8,67.7,41.5,0,Low,1,0,3.89
3,4,Male,Jaipur,Tier1,CS,AI,No,No,7.67,8.03,...,0,69.8,2.7,66.9,48.0,0,Mid,1,0,8.37
4,5,Male,Pune,Tier2,IT,DataScience,Yes,No,8.14,8.97,...,1,73.1,2.1,71.7,61.7,1,High,1,0,18.99


In [41]:
print(df.columns.tolist())

['StudentID', 'Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'SGPA_Sem1', 'SGPA_Sem2', 'SGPA_Sem3', 'SGPA_Sem4', 'SGPA_Sem5', 'SGPA_Sem6', 'SGPA_Sem7', 'SGPA_Sem8', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'CGPA_Tier', 'PlacementStatus', 'IsAnomaly', 'Salary Package']


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   StudentID           50000 non-null  int64  
 1   Gender              50000 non-null  object 
 2   City                50000 non-null  object 
 3   CollegeTier         50000 non-null  object 
 4   Stream              50000 non-null  object 
 5   Specialisation      50000 non-null  object 
 6   Hostel              50000 non-null  object 
 7   HistoryOfBacklogs   50000 non-null  object 
 8   SGPA_Sem1           50000 non-null  float64
 9   SGPA_Sem2           50000 non-null  float64
 10  SGPA_Sem3           50000 non-null  float64
 11  SGPA_Sem4           50000 non-null  float64
 12  SGPA_Sem5           50000 non-null  float64
 13  SGPA_Sem6           50000 non-null  float64
 14  SGPA_Sem7           50000 non-null  float64
 15  SGPA_Sem8           50000 non-null  float64
 16  CGPA

In [42]:
print(df.isnull().sum())

StudentID                0
Gender                   0
City                     0
CollegeTier              0
Stream                   0
Specialisation           0
Hostel                   0
HistoryOfBacklogs        0
SGPA_Sem1                0
SGPA_Sem2                0
SGPA_Sem3                0
SGPA_Sem4                0
SGPA_Sem5                0
SGPA_Sem6                0
SGPA_Sem7                0
SGPA_Sem8                0
CGPA                     0
AttendancePercent        0
Internships              0
Projects                 0
Workshops             4488
Certifications           0
Publications             0
AptitudeTestScore     4029
SoftSkillsRating      3526
CodingTestScore       2976
MockInterviewScore    4957
ExtraCurricular          0
CGPA_Tier                0
PlacementStatus          0
IsAnomaly                0
Salary Package           0
dtype: int64


In [ ]:
print("PlacementStatus:")
print(df["PlacementStatus"].value_counts())

print("\nCGPA_Tier:")
print(df["CGPA_Tier"].value_counts())

In [ ]:
exclude_cols = [
    "PlacementStatus",
    "CGPA_Tier",
    "Salary Package",
    "StudentID",
    "IsAnomaly"
]

X = df.drop(columns=exclude_cols, errors="ignore")

print("Number of features:", X.shape[1])
print(X.columns.tolist())

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [ ]:
X_binary = df.drop(
    columns=[
        "PlacementStatus",
        "CGPA_Tier",
        "Salary Package",
        "StudentID",
        "IsAnomaly"
    ],
    errors="ignore"
)

y_binary = df["PlacementStatus"]

In [ ]:
X_train_bin, X_val_bin, y_train_bin, y_val_bin = train_test_split(
    X_binary,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print("Training samples:", X_train_bin.shape[0])
print("Validation samples:", X_val_bin.shape[0])

In [ ]:
binary_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_val, y_val):

    # Fit the model
    model.fit(X_train, y_train)

    # Predictions
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    # Accuracy
    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    print("=" * 70)
    print("MODEL EVALUATION")
    print("=" * 70)

    print(f"\nTrain Accuracy: {train_accuracy:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")

    print("\nClassification Report - Validation Data")
    print("-" * 70)
    print(classification_report(y_val, val_pred))

    print("\nConfusion Matrix - Validation Data")
    print("-" * 70)
    print(confusion_matrix(y_val, val_pred))

    return {
        "model": model,
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "train_predictions": train_pred,
        "val_predictions": val_pred
    }

In [ ]:
binary_result = evaluate_classifier(
    binary_lr,
    X_train_bin,
    y_train_bin,
    X_val_bin,
    y_val_bin
)

In [ ]:
X_multi = df.drop(
    columns=[
        "PlacementStatus",
        "CGPA_Tier",
        "Salary Package",
        "StudentID",
        "IsAnomaly"
    ],
    errors="ignore"
)

y_multi = df["CGPA_Tier"]

In [ ]:
X_train_multi, X_val_multi, y_train_multi, y_val_multi = train_test_split(
    X_multi,
    y_multi,
    test_size=0.20,
    random_state=42,
    stratify=y_multi
)

print("Training samples:", X_train_multi.shape[0])
print("Validation samples:", X_val_multi.shape[0])

In [ ]:
multi_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            solver="lbfgs",
            random_state=42
        ))
    ]
)

In [ ]:
multi_lr = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            random_state=42
        ))
    ]
)

In [ ]:
multi_result = evaluate_classifier(
    multi_lr,
    X_train_multi,
    y_train_multi,
    X_val_multi,
    y_val_multi
)

In [ ]:
academic_features = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
]

academic_features = [
    col for col in academic_features
    if col in df.columns
]

print("Academic features:")
print(academic_features)

In [ ]:
X_academic = df[academic_features]
y_academic = df["PlacementStatus"]

In [ ]:
X_train_acad, X_val_acad, y_train_acad, y_val_acad = train_test_split(
    X_academic,
    y_academic,
    test_size=0.20,
    random_state=42,
    stratify=y_academic
)

In [ ]:
academic_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [ ]:
academic_lr = Pipeline(
    steps=[
        ("preprocessor", academic_preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [ ]:
academic_result = evaluate_classifier(
    academic_lr,
    X_train_acad,
    y_train_acad,
    X_val_acad,
    y_val_acad
)

In [ ]:
results_df = pd.DataFrame({
    "Model": [
        "Binary PlacementStatus LR",
        "Multinomial CGPA_Tier LR",
        "Academic PlacementStatus LR"
    ],
    "Train Accuracy": [
        binary_result["train_accuracy"],
        multi_result["train_accuracy"],
        academic_result["train_accuracy"]
    ],
    "Validation Accuracy": [
        binary_result["val_accuracy"],
        multi_result["val_accuracy"],
        academic_result["val_accuracy"]
    ]
})

results_df

In [ ]:
print(results_df.to_string(index=False))

In [ ]:
feature_names_binary = (
    binary_lr
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

coefficients_binary = (
    binary_lr
    .named_steps["classifier"]
    .coef_[0]
)

coef_binary_df = pd.DataFrame({
    "Feature": feature_names_binary,
    "Coefficient": coefficients_binary
})

coef_binary_df["AbsoluteCoefficient"] = (
    coef_binary_df["Coefficient"].abs()
)

coef_binary_df = coef_binary_df.sort_values(
    "AbsoluteCoefficient",
    ascending=False
)

coef_binary_df.head(20)

In [ ]:
top20_binary = coef_binary_df.head(20).sort_values(
    "Coefficient"
)

plt.figure(figsize=(10, 8))

plt.barh(
    top20_binary["Feature"],
    top20_binary["Coefficient"]
)

plt.xlabel("Logistic Regression Coefficient")
plt.ylabel("Feature")
plt.title("Top 20 Coefficients — PlacementStatus Logistic Regression")

plt.tight_layout()
plt.show()

In [ ]:
feature_names_multi = (
    multi_lr
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

multi_coefficients = (
    multi_lr
    .named_steps["classifier"]
    .coef_
)

print("Coefficient matrix shape:")
print(multi_coefficients.shape)

In [ ]:
classes = multi_lr.named_steps["classifier"].classes_

for i, class_label in enumerate(classes):

    coef_df = pd.DataFrame({
        "Feature": feature_names_multi,
        "Coefficient": multi_coefficients[i]
    })

    coef_df["AbsoluteCoefficient"] = (
        coef_df["Coefficient"].abs()
    )

    coef_df = coef_df.sort_values(
        "AbsoluteCoefficient",
        ascending=False
    )

    top20 = coef_df.head(20).sort_values(
        "Coefficient"
    )

    plt.figure(figsize=(10, 8))

    plt.barh(
        top20["Feature"],
        top20["Coefficient"]
    )

    plt.xlabel("Coefficient")
    plt.ylabel("Feature")
    plt.title(
        f"Top 20 Coefficients — CGPA_Tier Class {class_label}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
coef_plot = coef_binary_df.head(15).sort_values(
    "Coefficient"
)

plt.figure(figsize=(10, 7))

plt.barh(
    coef_plot["Feature"],
    coef_plot["Coefficient"]
)

plt.xlabel("Coefficient")
plt.title(
    "Top Logistic Regression Features for Placement Prediction"
)

plt.tight_layout()
plt.show()

In [ ]:
print("\n\nMODEL 1: BINARY PLACEMENT STATUS")
binary_result = evaluate_classifier(
    binary_lr,
    X_train_bin,
    y_train_bin,
    X_val_bin,
    y_val_bin
)


print("\n\nMODEL 2: MULTINOMIAL CGPA TIER")
multi_result = evaluate_classifier(
    multi_lr,
    X_train_multi,
    y_train_multi,
    X_val_multi,
    y_val_multi
)


print("\n\nMODEL 3: ACADEMIC PLACEMENT STATUS")
academic_result = evaluate_classifier(
    academic_lr,
    X_train_acad,
    y_train_acad,
    X_val_acad,
    y_val_acad
)